# Optymalizacja layoutu farmy wiatrowej — scipy (gradient-based)

Notebook demonstruje:
1. Budowę farmy 3×3 na siatce regularnej
2. Symulację śladów wake (wąski bin: jeden kierunek wiatru)
3. Optymalizację layoutu metodą scipy SLSQP
4. Porównanie śladów wake przed i po optymalizacji

**Projekt:** MFW Bałtyk 2 — Temat 2 (Lokalizacja i rozmieszczenie farm wiatrowych)

## 1. Importy i konfiguracja środowiska

In [ ]:
import sys
import os

# Dodaj katalog src/ do ścieżki, żeby Python znalazł nasze moduły
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))
# Jeśli notebook jest uruchomiony z folderu notebooks/, ścieżka wyżej jest poprawna.
# Jeśli z głównego katalogu projektu:
if not os.path.exists(os.path.join(sys.path[0], 'farm_model.py')):
    sys.path.insert(0, 'src')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time

from floris import TimeSeries
from floris.optimization.layout_optimization.layout_optimization_scipy import (
    LayoutOptimizationScipy,
)

from farm_model import FarmModel

# Ładna grafika w notebooku
%matplotlib inline
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

print('Importy OK')

## 2. Parametry symulacji

Tutaj ustawiamy wszystkie zmienne — turbinę, model wake, wiatr i rozmiar farmy.
Zmień te wartości, żeby eksperymentować.

In [ ]:
# ---------- Turbina i model wake ----------
TURBINE     = 'iea_15MW'   # Turbina 15 MW (offshore, bliska SG 14-222 DD)
WAKE_MODEL  = 'gch'        # Gauss Curl Hybrid — dobry balans dokładność/czas

# ---------- Layout startowy (siatka 3×3) ----------
N_ROWS      = 3
N_COLS      = 3
SPACING_D   = 7.0          # rozstaw turbin = 7 × średnica rotora

# ---------- Wąski bin wiatru (narrow wind bin) ----------
# Wąski bin = jeden kierunek i jedna prędkość.
# To uproszczenie: zamiast całej róży wiatrów (np. 36 kierunków × 20 prędkości = 720 warunków)
# używamy JEDNEGO warunka. Optymalizacja jest ~100× szybsza, ale wynik
# dotyczy tylko tego konkretnego wiatru — nie rocznej produkcji.
WIND_DIRECTION = 270.0     # zachód (wiatr wieje ze wschodu → turb. w rzędach prostopadle)
WIND_SPEED     = 9.0       # m/s — typowa prędkość powyżej rated dla 15 MW
TURB_INTENSITY = 0.06      # 6% intensywność turbulencji (morska, niski TI)

# ---------- Scipy SLSQP ----------
MAXITER     = 50           # max liczba iteracji optymalizatora
MIN_DIST_D  = 3.0          # minimalna odległość między turbinami [×D]

print(f'Turbina:      {TURBINE}')
print(f'Model wake:   {WAKE_MODEL}')
print(f'Layout:       {N_ROWS}×{N_COLS} = {N_ROWS*N_COLS} turbin, rozstaw {SPACING_D}D')
print(f'Wiatr:        {WIND_DIRECTION}° / {WIND_SPEED} m/s / TI={TURB_INTENSITY*100:.0f}%')
print(f'Scipy:        max {MAXITER} iteracji, min_dist={MIN_DIST_D}D')

## 3. Budowa modelu farmy

`FarmModel` to nasz wrapper na `FlorisModel` z NREL FLORIS v4.  
Obsługuje generatory layoutów, przełączanie modeli wake i wizualizację śladów.

In [ ]:
# Utwórz model farmy z wybraną turbiną i modelem wake
farm = FarmModel(wake_model=WAKE_MODEL, turbine=TURBINE)

# Wygeneruj siatkę 3×3
farm.set_layout_grid(n_rows=N_ROWS, n_cols=N_COLS, spacing_D=SPACING_D)

# Pokaż parametry fizyczne
D  = farm.D           # średnica rotora [m]
Hh = farm.hub_height  # wysokość piasty [m]

print(f'Liczba turbin:   {farm.n_turbines}')
print(f'Średnica rotora: D = {D} m')
print(f'Wysokość piasty: {Hh} m')
print(f'Rozstaw:         {SPACING_D}×D = {SPACING_D*D:.0f} m')
print(f'\nWspółrzędne X [m]: {farm.layout_x}')
print(f'Współrzędne Y [m]: {farm.layout_y}')

## 4. Dane wiatrowe — wąski bin

**`TimeSeries`** w FLORIS reprezentuje „serię czasową” warunków wiatrowych.  
Kiedy przekazujemy JEDEN kierunek i JEDNĄ prędkość — to jest właśnie **wąski bin**.

Wąski bin vs pełna róża wiatrów:

| Podejście | Warunki | Czas FLORIS | Kiedy używać |
|-----------|---------|-------------|---------------|
| Wąski bin (TimeSeries) | 1 | ~0.01 s | optymalizacja, debugowanie |
| Pełna róża (WindRose) | 36×20 = 720 | ~7 s | obliczenie AEP rocznego |

In [ ]:
# TimeSeries z jednym wektorem wiatru = wąski bin
# FLORIS interpretuje to jako jedną chwilę czasu (jeden „findex")
wind_data = TimeSeries(
    wind_directions   = np.array([WIND_DIRECTION]),  # [deg] — jeden kierunek
    wind_speeds       = np.array([WIND_SPEED]),       # [m/s] — jedna prędkość
    turbulence_intensities = np.array([TURB_INTENSITY]),
)

farm.set_wind_data(wind_data)

# Uruchom symulację przed optymalizacją
farm.run()

# Moc farmy w tym jednym warunku [MW]
power_before = farm.get_farm_power_mw()[0]   # get_farm_power_mw zwraca tablicę per findex
print(f'Moc farmy (przed optymalizacją): {power_before:.2f} MW')
print(f'Moc 1 turbiny bez wake:          {power_before / farm.n_turbines:.2f} MW')

# Moc per turbina [MW]
powers_per_turbine = farm.get_turbine_powers_mw()[0]  # findex=0
print(f'\nMoc per turbina [MW]:')
for i, p in enumerate(powers_per_turbine):
    row = i // N_COLS
    col = i % N_COLS
    print(f'  T{i} (rząd {row}, kol {col}): {p:.2f} MW')

**Obserwacja:** Turbiny w rzędach dalszych od wiatru mają mniejszą moc — są w cieniu (wake) turbin przed nimi.  
Przy WD=270° (wiatr z zachodu) wiatr wieje wzdłuż osi X, więc kolumny T0/T3/T6, T1/T4/T7, T2/T5/T8 tworzą rzędy prostopadłe do wiatru — każda kolumna jest za poprzednią.

## 5. Wizualizacja śladu wake — PRZED optymalizacją

In [ ]:
# plot_flow_field() oblicza poziomy przekrój pola prędkości na wysokości piasty.
# Ciemne obszary = mała prędkość = ślad wake za turbiną.
fig_before = farm.plot_flow_field(
    wind_direction = WIND_DIRECTION,
    wind_speed     = WIND_SPEED,
    ti             = TURB_INTENSITY,
    height         = Hh,                              # wysokość cięcia = piasta
    title          = f'PRZED optymalizacją | {N_ROWS}×{N_COLS} siatka | '
                     f'WD={WIND_DIRECTION}° WS={WIND_SPEED} m/s',
    show_rotors    = True,
    show_labels    = True,
    show_wind_arrow= True,
    figsize        = (13, 5),
)
plt.tight_layout()
plt.show()

print(f'\nMoc farmy PRZED: {power_before:.2f} MW')
print(f'(siatka regularna — turbiny ustawione wprost za sobą)')

## 6. Optymalizacja scipy (SLSQP)

### Jak działa SLSQP?

**SLSQP** (Sequential Least Squares Programming) to metoda gradientowa.  
Na każdym kroku szuka kierunku, w którym AEP rośnie najszybciej.

Gradient jest liczony **numerycznie** przez różnice skończone:  
dla każdej z `2N` współrzędnych (X i Y każdej turbiny) FLORIS jest uruchamiany osobno  
z małym perturbowaniem tej współrzędnej: `∂AEP/∂xᵢ ≈ (AEP(xᵢ+δ) − AEP(xᵢ)) / δ`

Koszt jednej iteracji = **2N wywołań FLORIS** (N=9 turbin → 18 wywołań/iter).  
Dla 50 iteracji = ~900 wywołań — stąd czas rzędu sekund/minut.

In [ ]:
# Zapamiętaj layout startowy (do porównania po)
x_init = farm.layout_x.copy()
y_init = farm.layout_y.copy()

# Granice obszaru — prostokąt obejmujący farmę + 1D margines z każdej strony
# (turbiny mogą wychodzić poza siatkę, ale nie poza ten prostokąt)
margin = 1.0 * D
x_min = x_init.min() - margin
x_max = x_init.max() + margin
y_min = y_init.min() - margin
y_max = y_init.max() + margin

# Granice jako wielokąt (lista wierzchołków) — wymagany format FLORIS
boundaries = [
    (x_min, y_min),
    (x_max, y_min),
    (x_max, y_max),
    (x_min, y_max),
]

min_dist = MIN_DIST_D * D   # minimalna odległość między turbinami [m]

print(f'Obszar optymalizacji: {x_max-x_min:.0f} m × {y_max-y_min:.0f} m')
print(f'Min. odległość:       {min_dist:.0f} m = {MIN_DIST_D}×D')
print(f'Max iteracji SLSQP:   {MAXITER}')
print(f'\nUruchamiam optymalizację...')

t0 = time.perf_counter()

# LayoutOptimizationScipy to klasa FLORIS, która owija scipy.optimize.minimize
# z metodą SLSQP i ograniczeniami geometrycznymi (boundaries + min_dist).
opt = LayoutOptimizationScipy(
    farm.fmodel,                           # FlorisModel — FLORIS ocenia AEP
    boundaries,                            # granice obszaru (wielokąt)
    min_dist    = min_dist,                # min. odległość między turbinami
    optOptions  = {                        # opcje dla scipy.optimize.minimize
        'maxiter': MAXITER,
        'disp'   : True,                   # wypisuj postęp
    },
)

# opt.optimize() blokuje — czeka na zakończenie wszystkich iteracji
solution = opt.optimize()

elapsed = time.perf_counter() - t0

# Wyniki: solution = (lista_x, lista_y)
x_opt = np.array(solution[0])
y_opt = np.array(solution[1])

print(f'\nOptymalizacja zakończona w {elapsed:.1f} s')

## 7. Wyniki optymalizacji

In [ ]:
# Ustaw zoptymalizowany layout w modelu i uruchom ponownie
farm.set_layout_custom(x_opt, y_opt, name='scipy_optimized')
farm.fmodel.set(wind_data=wind_data)   # przywróć dane wiatrowe (set() je czyści)
farm.run()

# Moc po optymalizacji
power_after = farm.get_farm_power_mw()[0]
improvement  = (power_after - power_before) / power_before * 100

print('=' * 50)
print(f'  Moc PRZED:      {power_before:.2f} MW')
print(f'  Moc PO:         {power_after:.2f} MW')
print(f'  Poprawa:        +{improvement:.2f}%')
print(f'  Czas scipy:     {elapsed:.1f} s')
print('=' * 50)

# Przesunięcia turbin
print('\nPrzesunięcia turbin [m]:')
print(f'  {"T":>3}  {"ΔX":>8}  {"ΔY":>8}  {"dist":>8}')
for i in range(len(x_init)):
    dx = x_opt[i] - x_init[i]
    dy = y_opt[i] - y_init[i]
    dist = np.hypot(dx, dy)
    print(f'  T{i:1d}  {dx:+8.1f}  {dy:+8.1f}  {dist:8.1f}')

## 8. Wizualizacja śladu wake — PO optymalizacji

In [ ]:
fig_after = farm.plot_flow_field(
    wind_direction = WIND_DIRECTION,
    wind_speed     = WIND_SPEED,
    ti             = TURB_INTENSITY,
    height         = Hh,
    title          = f'PO optymalizacji (scipy SLSQP) | '
                     f'WD={WIND_DIRECTION}° WS={WIND_SPEED} m/s | +{improvement:.1f}%',
    show_rotors    = True,
    show_labels    = True,
    show_wind_arrow= True,
    figsize        = (13, 5),
)
plt.tight_layout()
plt.show()

print(f'Moc farmy PO: {power_after:.2f} MW  (+{improvement:.2f}%)')
print(f'\nObserwacja: turbiny przesunęły się tak, żeby rzadziej nakrywać się wzajemnie śladami.')

## 9. Porównanie side-by-side: przed / po

Lewy panel: **layout i ślady przed optymalizacją** (siatka regularna, ślady wprost na kolejne turbiny).  
Prawy panel: **layout i ślady po optymalizacji** (turbiny przesunięte — mniejsze zachodzenie śladów).  
Środkowy panel: **strzałki przesunięcia** — każda turbina poruszyła się z czerwonego punktu do zielonego.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# ---- Panel 1: layout PRZED ----
ax = axes[0]
ax.scatter(x_init, y_init, s=100, c='#c8531a', edgecolors='white',
           linewidths=1.5, zorder=5, label='Przed')

# Numery turbin
for i, (x, y) in enumerate(zip(x_init, y_init)):
    ax.annotate(f'T{i}', (x, y), textcoords='offset points',
                xytext=(6, 6), fontsize=9, color='#333')

# Granice obszaru
bx = [b[0] for b in boundaries] + [boundaries[0][0]]
by = [b[1] for b in boundaries] + [boundaries[0][1]]
ax.plot(bx, by, 'k--', linewidth=1, alpha=0.4, label='Granice')

ax.set_title(f'PRZED\nMoc = {power_before:.1f} MW', fontsize=12)
ax.set_xlabel('X [m]')
ax.set_ylabel('Y [m]')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# ---- Panel 2: strzałki przesunięcia ----
ax = axes[1]
ax.scatter(x_init, y_init, s=70, c='#c8531a', alpha=0.5, zorder=4, label='Przed')
ax.scatter(x_opt,  y_opt,  s=100, c='#1e5c3a', edgecolors='white',
           linewidths=1.5, zorder=5, label='Po')

# Strzałki pokazujące przesunięcie każdej turbiny
for i in range(len(x_init)):
    dx = x_opt[i] - x_init[i]
    dy = y_opt[i] - y_init[i]
    if np.hypot(dx, dy) > 0.5:   # nie rysuj znikomo małych
        ax.annotate('',
            xy     = (x_opt[i],  y_opt[i]),
            xytext = (x_init[i], y_init[i]),
            arrowprops=dict(arrowstyle='->', color='#534AB7', lw=2.0, alpha=0.8),
        )

ax.plot(bx, by, 'k--', linewidth=1, alpha=0.4)
ax.set_title(f'Przesunięcia turbin\n(czerwony → zielony)', fontsize=12)
ax.set_xlabel('X [m]')
ax.set_aspect('equal')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# ---- Panel 3: layout PO ----
ax = axes[2]
ax.scatter(x_opt, y_opt, s=100, c='#1e5c3a', edgecolors='white',
           linewidths=1.5, zorder=5, label='Po')

for i, (x, y) in enumerate(zip(x_opt, y_opt)):
    ax.annotate(f'T{i}', (x, y), textcoords='offset points',
                xytext=(6, 6), fontsize=9, color='#333')

ax.plot(bx, by, 'k--', linewidth=1, alpha=0.4)
ax.set_title(f'PO  (+{improvement:.1f}%)\nMoc = {power_after:.1f} MW', fontsize=12)
ax.set_xlabel('X [m]')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

fig.suptitle(
    f'Optymalizacja scipy SLSQP | {farm.turbine_info["name"]} | '
    f'{N_ROWS}×{N_COLS} turbin | WD={WIND_DIRECTION}° WS={WIND_SPEED} m/s | '
    f'Czas: {elapsed:.1f} s',
    fontsize=12, fontweight='500',
)
fig.tight_layout()
plt.show()

## 10. Podsumowanie i kontekst dla przyspieszenia

### Co widzieliśmy

| | Wartość |
|---|---|
| Turbina | IEA 15 MW |
| Model wake | GCH (Gauss Curl Hybrid) |
| Wiatr (wąski bin) | WD=270°, WS=9 m/s, TI=6% |
| Layout startowy | 3×3 siatka, rozstaw 7D |
| Iteracje SLSQP | max 50 |

### Dlaczego scipy jest wolne?

Na każdej iteracji SLSQP musi obliczyć **gradient AEP** po wszystkich 2N zmiennych  
(X i Y każdej z N turbin). FLORIS nie ma gradientu analitycznego, więc scipy liczy  
go numerycznie — **2N wywołań FLORIS na iterację**:

```
∂AEP/∂xᵢ ≈ (AEP(xᵢ + δ) − AEP(xᵢ)) / δ
```

Dla 9 turbin i 50 iteracji = **~900 wywołań FLORIS**.

### Jak przyspieszyć (do dyskusji z profesorem)

**Opcja A — `ParFlorisModel`** (FLORIS zrównoleglenie wbudowane):
```python
from floris import ParFlorisModel
pfm = ParFlorisModel(config, max_workers=4)  # 4 rdzenie
# ParFlorisModel działa jako zamiennik FlorisModel w LayoutOptimizationScipy
opt = LayoutOptimizationScipy(pfm, boundaries, ...)
```
Każde wywołanie FLORIS rozkłada obliczenia warunków wiatrowych na kilka procesów.

**Opcja B — Multi-start** (wiele punktów startowych, każdy w osobnym procesie):
```python
from concurrent.futures import ProcessPoolExecutor

def one_start(seed):
    # perturbuj layout startowy losowo i uruchom scipy
    ...

with ProcessPoolExecutor(max_workers=4) as ex:
    results = list(ex.map(one_start, [42, 123, 456, 789]))
best = max(results, key=lambda r: r['power'])
```
Omija lokalne minima — scipy jest deterministyczne i tkwi w pierwszym minimum.

In [ ]:
# Podsumowanie liczbowe — gotowe do pokazania na spotkaniu
print('═' * 55)
print(f'  WYNIKI OPTYMALIZACJI SCIPY SLSQP')
print('═' * 55)
print(f'  Turbina:           {farm.turbine_info["name"]}')
print(f'  Model wake:        {WAKE_MODEL.upper()} (Gauss Curl Hybrid)')
print(f'  Bin wiatru:        WD={WIND_DIRECTION}°, WS={WIND_SPEED} m/s')
print(f'  Layout startowy:   {N_ROWS}×{N_COLS} siatka, rozstaw {SPACING_D}D = {SPACING_D*D:.0f} m')
print(f'  Turbiny:           {farm.n_turbines}')
print('─' * 55)
print(f'  Moc PRZED:         {power_before:.2f} MW')
print(f'  Moc PO:            {power_after:.2f} MW')
print(f'  Poprawa:           +{improvement:.2f}%  (+{power_after-power_before:.2f} MW)')
print('─' * 55)
print(f'  Czas optymalizacji: {elapsed:.1f} s')
print(f'  Max iteracji SLSQP: {MAXITER}')
print(f'  Min. odległość:     {MIN_DIST_D}×D = {min_dist:.0f} m')
print('═' * 55)